# Mat & Comp – Lección 2: El azar
## Lanzar monedas con Python y el triángulo de Sierpinski

**Curso:** Matemáticas y Computación · **Duración sugerida:** 2 sesiones de 50 minutos

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Usar el módulo `random` para simular monedas y dados (`random.choice`, `random.randint`, `random.random`) y repetir un experimento con `random.seed`.
2. Repetir un experimento miles de veces y **contar** resultados para estimar probabilidades.
3. Ver con gráficas la **ley de los grandes números**: la proporción de caras se acerca a 1/2 cuando lanzamos muchas veces.
4. Descubrir que el azar produce **rachas** más largas de lo que nuestra intuición espera.
5. Usar **sliders interactivos** (`ipywidgets`) para explorar cómo el **juego del caos** dibuja el triángulo de Sierpinski, punto a punto.

## 🔍 ¿Puedes fingir el azar?

**Experimento antes de abrir Python.** Escribe en un papel una lista de **100 lanzamientos de moneda inventados**, por ejemplo `CSSCSCCS…` (C = cara, S = sello). Intenta que parezca de verdad aleatoria.

Después, mira tu lista: ¿cuál es la **racha más larga** de resultados iguales seguidos?

✏️ *Escribe aquí tu racha más larga. Al final de la sección 4 la compararemos con monedas de verdad (simuladas por la computadora).*

Y una pregunta más extraña para el final de la lección: **¿puede el azar dibujar una figura perfectamente ordenada?**

## 📚 El concepto: experimentos aleatorios y probabilidad

Un **experimento aleatorio** es algo cuyo resultado no podemos predecir antes de hacerlo: lanzar una moneda, un dado, sacar una carta.

La **probabilidad** de un resultado mide qué tan frecuente es cuando repetimos el experimento muchas veces:

$$P(\text{cara}) = \frac{\text{casos favorables}}{\text{casos posibles}} = \frac{1}{2}$$

Con la computadora podemos lanzar una moneda **un millón de veces** en un segundo y comparar lo que pasa con lo que predice la teoría.

## 1. El módulo `random`

In [ ]:
import random

print(random.random())            # un número decimal al azar entre 0 y 1
print(random.randint(1, 6))       # un entero al azar entre 1 y 6 (¡un dado!)
print(random.choice(["cara", "sello"]))   # elige un elemento de la lista

Ejecuta la celda anterior varias veces (Ctrl + Enter). ¿Salen siempre los mismos resultados?

In [ ]:
# random.seed fija la "semilla": así todos obtienen la misma secuencia "aleatoria"
random.seed(2026)
print([random.randint(1, 6) for _ in range(10)])

random.seed(2026)
print([random.randint(1, 6) for _ in range(10)])   # ¡igual que la anterior!

Las computadoras no lanzan monedas de verdad: usan una fórmula que produce números que **parecen** aleatorios (se llaman **pseudoaleatorios**). La semilla es el punto de partida de esa fórmula.

**¿Qué pasa si…?**
- …cambias la semilla `2026` por tu fecha de cumpleaños? ¿Y si quitas las líneas con `seed`?

## 2. Lanzando una moneda

In [ ]:
def lanzar_moneda():
    """Devuelve 'C' (cara) o 'S' (sello) con la misma probabilidad."""
    return random.choice("CS")

lanzamientos = [lanzar_moneda() for _ in range(20)]
print("".join(lanzamientos))
print("Caras:", lanzamientos.count("C"))
print("Sellos:", lanzamientos.count("S"))

In [ ]:
# Lanzamos cada vez más monedas: ¿se acerca la proporción de caras a 1/2?
for n in [10, 100, 1000, 10000, 100000, 1000000]:
    caras = sum(1 for _ in range(n) if lanzar_moneda() == "C")
    print(f"{n:>8} lanzamientos → proporción de caras = {caras / n:.4f}")

Veámoslo en una gráfica. Después de cada lanzamiento calculamos la proporción de caras **acumulada** hasta ese momento. Mueve el **slider** para cambiar el número de lanzamientos y el número de experimentos (cada uno es una línea de color).

In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider

def proporcion_acumulada(n):
    """Lista con la proporción de caras después de 1, 2, ..., n lanzamientos."""
    caras = 0
    proporciones = []
    for i in range(1, n + 1):
        if lanzar_moneda() == "C":
            caras = caras + 1
        proporciones.append(caras / i)
    return proporciones

def grafica_proporcion(lanzamientos, experimentos):
    plt.figure(figsize=(10, 4))
    for _ in range(experimentos):
        plt.plot(range(1, lanzamientos + 1), proporcion_acumulada(lanzamientos), lw=1)
    plt.axhline(0.5, color="black", ls="--", label="1/2")
    plt.ylim(0, 1)
    plt.xscale("log")
    plt.xlabel("número de lanzamientos (escala logarítmica)")
    plt.ylabel("proporción de caras")
    plt.title("Ley de los grandes números")
    plt.legend()
    plt.show()

interact(grafica_proporcion,
         lanzamientos=IntSlider(value=1000, min=10, max=20000, step=10, continuous_update=False),
         experimentos=IntSlider(value=5, min=1, max=20, continuous_update=False));

Esto es la **ley de los grandes números**: con pocos lanzamientos la proporción salta mucho, pero con muchos se estabiliza cerca de 1/2.

¡Cuidado! Eso **no** significa que después de 5 caras seguidas "toca" sello. La moneda no tiene memoria: cada lanzamiento es nuevo. Lo que pasa es que 5 caras de más se "diluyen" entre miles de lanzamientos.

## 3. ¿Cuántas caras salen en 10 lanzamientos?

Lanzamos 10 monedas y contamos las caras. Puede salir cualquier número entre 0 y 10, pero ¿todos son igual de probables? Repitamos el experimento muchas veces y hagamos un **diagrama de barras**.

In [ ]:
def caras_en(n):
    """Lanza n monedas y devuelve cuántas caras salieron."""
    return sum(1 for _ in range(n) if lanzar_moneda() == "C")

def histograma_caras(monedas, repeticiones):
    resultados = [caras_en(monedas) for _ in range(repeticiones)]
    conteo = [resultados.count(k) for k in range(monedas + 1)]
    plt.figure(figsize=(9, 4))
    plt.bar(range(monedas + 1), conteo, color="#4c8bf5")
    plt.xlabel(f"número de caras en {monedas} lanzamientos")
    plt.ylabel("veces que ocurrió")
    plt.title(f"{repeticiones} experimentos de {monedas} monedas")
    plt.show()
    mas_comun = conteo.index(max(conteo))
    print("Resultado más frecuente:", mas_comun, "caras")
    print("Veces que salieron", monedas, "caras de", monedas, ":", conteo[-1])

interact(histograma_caras,
         monedas=IntSlider(value=10, min=1, max=50, continuous_update=False),
         repeticiones=IntSlider(value=5000, min=100, max=20000, step=100, continuous_update=False));

Las barras forman una **campana**: los resultados del centro son mucho más probables que los extremos. ¿Por qué? Hay **una sola** manera de sacar 10 caras (CCCCCCCCCC), pero **252** maneras de sacar 5 caras y 5 sellos.

**¿Qué pasa si…?**
- …pones `monedas = 1`? ¿Y `monedas = 2`? ¿Por qué con 2 monedas el resultado "1 cara" sale el doble que "0 caras"?
- …subes `monedas` a 50? ¿La campana se vuelve más ancha o más angosta *comparada con el total*?

## 4. Rachas: el azar es más "rachoso" de lo que crees

Volvamos al experimento del principio. Calculemos la **racha más larga** en 100 lanzamientos reales (simulados).

In [ ]:
def racha_mas_larga(secuencia):
    """Longitud de la racha más larga de símbolos iguales seguidos."""
    mejor = 1
    actual = 1
    for i in range(1, len(secuencia)):
        if secuencia[i] == secuencia[i - 1]:
            actual = actual + 1          # la racha continúa
        else:
            actual = 1                   # empieza una racha nueva
        mejor = max(mejor, actual)
    return mejor

secuencia = "".join(lanzar_moneda() for _ in range(100))
print(secuencia)
print("Racha más larga:", racha_mas_larga(secuencia))

In [ ]:
# Repetimos 10 000 veces: ¿qué rachas son típicas en 100 lanzamientos?
rachas = [racha_mas_larga("".join(lanzar_moneda() for _ in range(100))) for _ in range(10000)]

plt.figure(figsize=(9, 4))
valores = range(1, max(rachas) + 1)
plt.bar(valores, [rachas.count(v) for v in valores], color="#f2a541")
plt.xlabel("racha más larga en 100 lanzamientos")
plt.ylabel("número de experimentos")
plt.title("¿Qué tan larga es la racha más larga?")
plt.show()

al_menos_6 = sum(1 for r in rachas if r >= 6) / len(rachas)
print(f"Probabilidad de una racha de 6 o más: {al_menos_6:.0%}")

En 100 lanzamientos de verdad, lo más común es una racha de **6 o 7**, y casi siempre hay alguna de 5 o más. Las personas que inventan lanzamientos casi nunca escriben rachas tan largas: ¡creen que "no parece aleatorio"!

Algunos profesores usan este truco: piden a la mitad del curso lanzar una moneda de verdad 100 veces y a la otra mitad inventarlo… y adivinan quién hizo trampa mirando las rachas.

In [ ]:
# Copia aquí tus 100 lanzamientos inventados (solo letras C y S) y compara
mis_lanzamientos = "CSCSSCSCCSCSSCSCSCCSSCSCSCSCCSCSSCSCSCCSSCSCSCCSCSSCSCSCCSCSSCSCSCSCCSSCSCSCCSCSSCSCSCCSCSSCSCSCSCCS"
print("Cantidad de lanzamientos:", len(mis_lanzamientos))
print("Caras:", mis_lanzamientos.count("C"))
print("Tu racha más larga:", racha_mas_larga(mis_lanzamientos))

## 5. El juego del caos

Ahora un experimento sorprendente, inventado por el matemático Michael Barnsley en 1988:

1. Dibuja los tres vértices de un triángulo: **A**, **B** y **C**.
2. Empieza en un punto cualquiera.
3. **Lanza un dado**: si sale 1 o 2 elige A; si sale 3 o 4, B; si sale 5 o 6, C.
4. Muévete **hasta la mitad del camino** hacia el vértice elegido y marca un punto.
5. Repite desde el paso 3… miles de veces.

✏️ *Predicción: después de muchísimos puntos, ¿qué figura crees que aparece? ¿Una mancha que llena el triángulo? ¿Algo más?*

In [ ]:
# Vértices del triángulo (un triángulo equilátero)
A = (0.0, 0.0)
B = (1.0, 0.0)
Cv = (0.5, 3**0.5 / 2)
vertices = [A, B, Cv]
nombres = ["A", "B", "C"]
colores = ["#e63946", "#2a9d8f", "#457b9d"]

def juego_del_caos(n, r=0.5, semilla=1):
    """Devuelve las listas xs, ys de n puntos y el vértice elegido en cada paso."""
    random.seed(semilla)
    x, y = 0.3, 0.6                     # punto de partida
    xs, ys, elegidos = [x], [y], [None]
    for _ in range(n):
        dado = random.randint(1, 6)
        v = (dado - 1) // 2             # 1,2 → 0 (A); 3,4 → 1 (B); 5,6 → 2 (C)
        vx, vy = vertices[v]
        x = x + r * (vx - x)            # avanzar una fracción r hacia el vértice
        y = y + r * (vy - y)
        xs.append(x); ys.append(y); elegidos.append(v)
    return xs, ys, elegidos

def dibujar_triangulo(ax):
    tx = [A[0], B[0], Cv[0], A[0]]
    ty = [A[1], B[1], Cv[1], A[1]]
    ax.plot(tx, ty, color="lightgray")
    for (vx, vy), nombre, col in zip(vertices, nombres, colores):
        ax.plot(vx, vy, "o", color=col, ms=10)
        ax.annotate(nombre, (vx, vy), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=13, color=col)
    ax.set_aspect("equal"); ax.axis("off")

xs, ys, elegidos = juego_del_caos(5)
for i in range(1, 6):
    print(f"Paso {i}: vértice {nombres[elegidos[i]]} → punto ({xs[i]:.3f}, {ys[i]:.3f})")

### Paso a paso

Mueve el slider **paso** para ver cómo se construye la secuencia: la línea punteada une el punto anterior con el vértice elegido, y el punto nuevo queda justo a la mitad.

In [ ]:
pasos_xs, pasos_ys, pasos_v = juego_del_caos(40)

def ver_paso(paso):
    fig, ax = plt.subplots(figsize=(6, 6))
    dibujar_triangulo(ax)
    ax.scatter(pasos_xs[:paso + 1], pasos_ys[:paso + 1], s=18, color="black", zorder=3)
    if paso > 0:
        v = pasos_v[paso]
        vx, vy = vertices[v]
        ax.plot([pasos_xs[paso - 1], vx], [pasos_ys[paso - 1], vy], "--", color=colores[v])
        ax.scatter([pasos_xs[paso]], [pasos_ys[paso]], s=80, color=colores[v], zorder=4)
        ax.set_title(f"Paso {paso}: el dado eligió {nombres[v]}")
    else:
        ax.set_title("Punto de partida")
    plt.show()

interact(ver_paso, paso=IntSlider(value=0, min=0, max=40, continuous_update=False));

### Muchos puntos

Ahora mueve el slider **puntos** desde unos pocos hasta 50 000. Cada punto tiene el color del vértice hacia el que saltó. ¿Qué aparece?

In [ ]:
todos_xs, todos_ys, todos_v = juego_del_caos(50000)
col_puntos = ["white"] + [colores[v] for v in todos_v[1:]]

def ver_puntos(puntos):
    fig, ax = plt.subplots(figsize=(7, 7))
    dibujar_triangulo(ax)
    # dejamos fuera los primeros 10 puntos, que aún están "llegando"
    inicio = 10 if puntos > 100 else 0
    ax.scatter(todos_xs[inicio:puntos + 1], todos_ys[inicio:puntos + 1],
               s=0.3 if puntos > 5000 else 4, c=col_puntos[inicio:puntos + 1])
    ax.set_title(f"Juego del caos con {puntos} puntos")
    plt.show()

interact(ver_puntos, puntos=IntSlider(value=50, min=10, max=50000, step=10, continuous_update=False));

¡Es el **triángulo de Sierpinski**! Un triángulo con un hueco triangular en el centro, y cada uno de los tres triángulos que quedan tiene a su vez su hueco… y así infinitamente. Una figura así, que se repite dentro de sí misma, se llama **fractal**.

Fíjate en los colores: todos los puntos rojos están en el triángulo pequeño de A, los verdes en el de B y los azules en el de C. ¿Por qué? Porque saltar a la mitad hacia A te deja siempre en la mitad del triángulo cercana a A.

**El azar, repetido muchas veces con una regla sencilla, produce un orden perfecto.** 🤯

### ¿Qué pasa si…?

- …saltamos una fracción distinta de 1/2? Mueve el slider **r**. ¿Qué pasa con r = 0,3? ¿Y con r = 0,7?
- …cambias la **semilla**? ¿Cambia la figura final?

In [ ]:
def ver_fraccion(r, semilla):
    xs, ys, vs = juego_del_caos(30000, r=r, semilla=semilla)
    fig, ax = plt.subplots(figsize=(6, 6))
    dibujar_triangulo(ax)
    ax.scatter(xs[10:], ys[10:], s=0.3, c=[colores[v] for v in vs[10:]])
    ax.set_title(f"Juego del caos con r = {r:.2f}")
    plt.show()

interact(ver_fraccion,
         r=FloatSlider(value=0.5, min=0.1, max=0.9, step=0.05, continuous_update=False),
         semilla=IntSlider(value=1, min=1, max=100, continuous_update=False));

Con r = 1/2 la figura **no depende de la semilla**: cada vez que juegas, el azar es distinto, pero el triángulo de Sierpinski es siempre el mismo. Con r más pequeño los triángulos se solapan y se llenan; con r más grande se separan en "polvo".

---
## ✍️ Ejercicios

Resuelve cada ejercicio en la celda de código que está debajo. Puedes reutilizar `lanzar_moneda`, `caras_en`, `racha_mas_larga` y `juego_del_caos` (si reiniciaste Colab, vuelve a ejecutar las celdas anteriores).

### Ejercicio 1 (básico) · Un dado justo
Lanza un dado 6000 veces con `random.randint(1, 6)` y cuenta cuántas veces sale cada cara.
¿Cuántas esperabas de cada una? Haz un diagrama de barras con `plt.bar`.

In [ ]:
# Escribe tu código aquí

### Ejercicio 2 (básico) · Suma de dos dados
Lanza **dos** dados y suma los resultados. Repite 10 000 veces y haz un diagrama de barras de las sumas (de 2 a 12).
¿Cuál es la suma más frecuente? ¿Por qué el 7 sale más que el 2? (Pista: cuenta de cuántas formas se puede sacar cada suma.)

In [ ]:
# Escribe tu código aquí

### Ejercicio 3 (intermedio) · Esperando la primera cara
Lanza una moneda hasta que salga la **primera cara** y cuenta cuántos lanzamientos necesitaste (usa `while`).
Repite el experimento 10 000 veces. ¿Cuál es el **promedio**? ¿Cuál fue la espera más larga?

In [ ]:
# Escribe tu código aquí

### Ejercicio 4 (intermedio) · Una moneda trucada
Escribe `moneda_trucada(p)` que dé cara con probabilidad p (pista: `random.random() < p`).
a) Con p = 0,7, comprueba con 100 000 lanzamientos que la proporción de caras se acerca a 0,7.
b) Un amigo te da una moneda y no sabes si está trucada. Lanzas 100 veces y salen 60 caras. Simula 10 000 veces 100 lanzamientos de una moneda **justa**: ¿qué tan seguido salen 60 caras o más? ¿Sospecharías de la moneda?

In [ ]:
# Escribe tu código aquí

### Ejercicio 5 (avanzado) · El juego del caos en un cuadrado
Cambia los vértices por los cuatro de un cuadrado: (0, 0), (1, 0), (1, 1), (0, 1), y elige uno al azar en cada paso con r = 1/2.
a) ¿Aparece un fractal o el cuadrado se llena?
b) Ahora agrega una regla: **no se puede elegir el mismo vértice dos veces seguidas**. ¿Qué figura aparece?

In [ ]:
# Escribe tu código aquí

---
## 🏆 Reto final (opcional) · El zoológico de fractales

1. **Pentágono:** usa los 5 vértices de un pentágono regular (pista: `(math.cos(2*math.pi*k/5), math.sin(2*math.pi*k/5))` para k = 0, …, 4) y r = 0,618. ¿Qué ves?
2. **Alfombra de Sierpinski:** usa 8 puntos de un cuadrado (las 4 esquinas y los 4 puntos medios de los lados) y r = 2/3.
3. Haz un slider que te deje elegir el **número de lados** del polígono y otro para **r**. Busca el valor de r que deja los pedazos "tocándose" sin solaparse.

In [ ]:
# Escribe tu código aquí

---
## 🎯 Cierre: lo que aprendiste hoy

- El módulo `random` simula monedas y dados; los números son **pseudoaleatorios** y `random.seed` permite repetir una secuencia.
- **Ley de los grandes números:** al repetir muchas veces, la proporción de caras se acerca a 1/2, aunque la moneda no tenga memoria.
- En 10 lanzamientos el resultado más probable es 5 caras: hay muchas más **formas** de obtenerlo, y las barras forman una **campana**.
- El azar real tiene **rachas largas**: en 100 lanzamientos lo normal es una racha de 6 o 7.
- El **juego del caos**, una regla sencilla repetida al azar, dibuja el **triángulo de Sierpinski**, un **fractal**.
- En Python aprendiste: `random`, `random.seed`, `count`, `join`, diagramas de barras y **sliders** con `ipywidgets.interact`.

**Para pensar:** si el resultado de cada lanzamiento es impredecible, ¿por qué el triángulo final es siempre el mismo? ¿Qué es lo que sí podemos predecir del azar?